# **Heart Stroke Prediction with Logistic Regression**

# 1.Project Summary: Health Outcome Prediction

# 2.What Does Logistic Regression Mean Here?
Logistic Regression is used as a binary classification model in this scenario. It helps us determine whether a patient falls into one of two categories:

0 — No Stroke: Patient is unlikely to have experienced a stroke.

1 — Stroke: Patient is likely to have experienced a stroke.

The model assigns a probability to the stroke outcome based on risk factors such as age, hypertension, heart disease, and glucose levels.

2. Environment Setup and Dataset Import

We start by getting our tools ready and loading the main dataset.

In [34]:
# !pip install pandas numpy scikit-learn matplotlib
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and Modeling Tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

In [35]:
# Load the stroke dataset into a DataFrame
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

# Check the contents of the dataset
print(df.info())
print("\nShowing the first few rows:")
print(df.head())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB
None

Showing the first few rows:
      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1    

#II. Taking a Closer Look at the Dataset

The first rule of data science is: Know Your Data!

A. The Imbalance Problem (A Crucial Discovery!)


One of the most notable aspects of this dataset is the disproportionate distribution of stroke cases.

In [36]:
# See how many people had a stroke vs. not (in percentage form)
stroke_counts = df['stroke'].value_counts(normalize=True)
print("Stroke Class Distribution:\n", stroke_counts)

# Create a bar chart to show the imbalance
plt.figure(figsize=(6, 4))
sns.countplot(x='stroke', data=df, palette='viridis')
plt.title('Distribution of Stroke Cases')
plt.xticks([0, 1], ['No Stroke (95.1%)', 'Stroke (4.9%)'])
plt.ylabel('Number of Records')
plt.xlabel('Stroke Outcome')
plt.grid(axis='y', linestyle='--')
plt.savefig('stroke_imbalance_distribution.png')
plt.close()


Stroke Class Distribution:
 stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64


/tmp/ipython-input-2699770889.py:7: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(x='stroke', data=df, palette='viridis')


### **B. Missing Values**

We checked the data earlier and found missing values in the bmi column.

In [37]:
print("\nMissing Values Count:")
print(df.isnull().sum())


Missing Values Count:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


### **Observation:** bmi has 201 missing values.

III. Preparing the Data and Designing Predictive Attributes.

Before we can train our model, we must clean, transform, and prepare the data.

# ** A. Data Tidying and Missing Value Handling.**


1) Drop 'id' and handle 'Other' Gender: The id column is a unique identifier and has no predictive power. The gender column has one outlier ('Other'), which is too few to be useful, so we'll drop that row.

2) Impute Missing BMI: Since bmi is a continuous numerical feature, we will fill its missing values with the median, as the median is less sensitive to outliers than the mean.

In [38]:
# 1. Drop 'id' and handle 'Other' gender
df_cleaned = df.drop('id', axis=1)
df_cleaned = df_cleaned[df_cleaned['gender'] != 'Other']

# 2. Impute missing 'bmi' values with the median
bmi_median = df_cleaned['bmi'].median()
df_cleaned['bmi'].fillna(bmi_median, inplace=True)

# Verification
print("Missing values after cleaning:")
print(df_cleaned.isnull().sum())

/tmp/ipython-input-727647313.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned['bmi'].fillna(bmi_median, inplace=True)


Missing values after cleaning:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


#B. Normalization of Variables and Categorical Data Transformation.

1) One-Hot Encoding: Logistic Regression requires all features to be numerical. We use One-Hot Encoding for categorical variables (like gender, work_type, smoking_status) to convert them into a set of binary (0 or 1) columns. We use drop_first=True to avoid multicollinearity.

2) Feature Scaling: Numerical features (age, avg_glucose_level, bmi) must be scaled. StandardScaler transforms the data to have a mean of $0$ and a standard deviation of $1$ (Z-score normalization). This prevents features with larger ranges from dominating the model.

In [39]:
# 1. Apply One-Hot Encoding to categorical variables
categorical_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
df_model = pd.get_dummies(df_cleaned, columns=categorical_cols, drop_first=True)

# 2. Separate features (X) and target variable (y)
X = df_model.drop('stroke', axis=1)
y = df_model['stroke']

# 3. Split the dataset into training and testing sets
# Using 'stratify=y' to maintain the same stroke proportion in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Standardize numerical features
numerical_cols = ['age', 'avg_glucose_level', 'bmi']
scaler = StandardScaler()

# Fit on training data, transform both training and testing data
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("\nTraining Features Shape:", X_train.shape)
print("Preview of scaled and encoded data:\n", X_train.head())



Training Features Shape: (4087, 15)
Preview of scaled and encoded data:
            age  hypertension  heart_disease  avg_glucose_level       bmi  \
845   0.209397             0              0          -0.821221  0.549234   
3745 -0.629845             0              0          -0.485884 -0.988900   
4184 -0.364822             0              0           0.302317 -0.769166   
3410 -0.232310             0              0           0.062342  0.497532   
284  -1.292405             0              0          -0.527297  0.355351   

      gender_Male  ever_married_Yes  work_type_Never_worked  \
845         False              True                   False   
3745        False             False                   False   
4184        False              True                   False   
3410         True              True                   False   
284          True             False                   False   

      work_type_Private  work_type_Self-employed  work_type_children  \
845               

# IV. Predictive Modeling Using Logistic Regression

**Time to build the predictive engine!**

A. Logistic Regression: Methodology

Logistic Regression models the probability of a binary event using the Sigmoid function (or Logistic function):$$\Large P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_n x_n)}}$$Where:$P(y=1)$ is the probability of a stroke (1).$\beta_i$ are the coefficients (weights) for each feature $x_i$, which represent the log-odds of a stroke.

**B. Training the Model with Class Balancing**

To combat the data imbalance (95% No Stroke vs. 5% Stroke), we use the class_weight='balanced' parameter. This tells the model to internally assign a higher penalty (weight) to misclassifying the minority class (stroke), forcing it to pay more attention to the rare cases.

In [40]:
# Create the Logistic Regression model, adjusting for imbalanced classes
log_reg = LogisticRegression(random_state=42, solver='liblinear', class_weight='balanced')
# Train the model using the training data
log_reg.fit(X_train, y_train)

# Make predictions on the test data
y_pred = log_reg.predict(X_test)
# Get the probability of stroke for each test sample (needed for ROC)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

# Classification Report
print("--- Classification Report ---")
report = classification_report(y_test, y_pred, target_names=['No Stroke (0)', 'Stroke (1)'])
print(report)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
                     index=['Actual No Stroke', 'Actual Stroke'],
                     columns=['Predicted No Stroke', 'Predicted Stroke'])
print("\n--- Confusion Matrix ---")
print(cm_df)

--- Classification Report ---
               precision    recall  f1-score   support

No Stroke (0)       0.99      0.73      0.84       972
   Stroke (1)       0.13      0.80      0.23        50

     accuracy                           0.74      1022
    macro avg       0.56      0.77      0.53      1022
 weighted avg       0.94      0.74      0.81      1022


--- Confusion Matrix ---
                  Predicted No Stroke  Predicted Stroke
Actual No Stroke                  712               260
Actual Stroke                      10                40


## **V. Model Evaluation and Interpretation**

Since accuracy is misleading here, we focus on more robust metrics.

### **A. Confusion Matrix and Classification Report**
Checking Model Predictions with a Confusion Matrix

In [41]:
# 1. Predictions
y_pred = log_reg.predict(X_test)
y_pred_proba = log_reg.predict_proba(X_test)[:, 1]

# 2. Classification Report
print("--- Classification Report ---")
report = classification_report(y_test, y_pred, target_names=['No Stroke (0)', 'Stroke (1)'])
print(report)

# 3. Confusion Matrix
print("\n--- Confusion Matrix (Actual vs. Predicted) ---")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Actual No Stroke', 'Actual Stroke'], columns=['Predicted No Stroke', 'Predicted Stroke'])
print(cm_df)

--- Classification Report ---
               precision    recall  f1-score   support

No Stroke (0)       0.99      0.73      0.84       972
   Stroke (1)       0.13      0.80      0.23        50

     accuracy                           0.74      1022
    macro avg       0.56      0.77      0.53      1022
 weighted avg       0.94      0.74      0.81      1022


--- Confusion Matrix (Actual vs. Predicted) ---
                  Predicted No Stroke  Predicted Stroke
Actual No Stroke                  712               260
Actual Stroke                      10                40


### **B. The ROC-AUC Curve (Overall Separability)**
The Receiver Operating Characteristic (ROC) Curve and its Area Under the Curve (AUC) score provide a single, powerful measure of the model's performance across all possible classification thresholds.

In [42]:
# 1. Compute ROC curve metrics and AUC score
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
print(f"\nROC-AUC Score: {roc_auc:.4f}")

# 2. Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig('roc_curve_logistic_regression.png')
plt.close()



ROC-AUC Score: 0.8390


### **Results & Interpretation**

### **The calculated ROC-AUC Score is $\mathbf{0.8390}$.**
Key Interpretation: An AUC of $\mathbf{0.84}$ is generally considered a strong result. It means that there is an 84% chance that the model will rank a randomly chosen positive instance (Stroke) higher than a randomly chosen negative instance (No Stroke). This confirms the model is highly effective at discriminating between the two classes.

## C. Model Feature Coefficients and Their Impact

Logistic Regression's best feature is its interpretability. The coefficients (weights) tell us how much each variable contributes to the log-odds of a stroke. A larger positive coefficient means a higher risk factor.

In [43]:
# Extract the model coefficients for each feature
coefficients = pd.Series(log_reg.coef_[0], index=X.columns)

# Convert coefficients to odds ratios
# Odds Ratio = exp(coefficient)
odds_ratios = np.exp(coefficients)

# Create a DataFrame combining coefficients and odds ratios, sorted by coefficient value
results = pd.DataFrame({
    'Coefficient': coefficients,
    'Odds_Ratio': odds_ratios
}).sort_values(by='Coefficient', ascending=False)

# Display the top 10 features by coefficient
print("\n--- Top 10 Feature Coefficients and Odds Ratios ---")
print(results.head(10))



--- Top 10 Feature Coefficients and Odds Ratios ---
                                Coefficient  Odds_Ratio
age                                1.879425    6.549735
work_type_children                 0.839779    2.315856
hypertension                       0.625876    1.869883
smoking_status_smokes              0.278821    1.321570
avg_glucose_level                  0.212196    1.236390
Residence_type_Urban               0.169548    1.184769
heart_disease                      0.142194    1.152801
smoking_status_formerly smoked     0.084984    1.088700
bmi                                0.058660    1.060415
work_type_Private                  0.013468    1.013559


## **VI. Conclusion and Key Takeaways**

The project demonstrates the use of Logistic Regression for predicting stroke outcomes on a dataset with uneven class distribution.